# 🗓️ 22일차 스터디 노트북 — 비재귀 퀵 정렬 · 퀵 정렬 개선 · 병합의 시작

**오늘 범위**: 06-6 후반 (비재귀 퀵 정렬 → 스택 크기 → 피벗 선택 → 시간 복잡도 → 개선판 `quick_sort2`) · 보충수업 6-4 `sorted()` · 06-7 병합 정렬 도입

## 난이도 태그
🟢 **기본** (전원 필수) / 🟡 **표준** (팀 목표선) / 🔴 **심화** (도전)

## 유형 태그
**[손]** 손으로 추적 · **[빈칸]** 빈칸 채우기 · **[예측]** 실행 전 결과 맞히기 · **[구현]** · **[디버깅]** · **[설명]** · **[실험]**

---

## 오늘의 두 가지 질문

> **Q1. "재귀를 스택으로 바꿨을 뿐인데, 왜 갑자기 `push` 순서를 고민해야 할까?"**
>
> **Q2. "21일차 퀵 정렬은 이미 O(n log n)인데, 교재는 왜 굳이 `sort3`와 삽입 정렬을 덧붙였을까?"**

오늘 노트북은 **앞부분(1~10번)에서 알고리즘의 원리를 뼈대까지 분해**하고,
**뒷부분(11~17번)에서 그 원리를 코드로 구현·응용**하는 순서로 짜여 있어.

> 📁 아래 **"부록: 오늘의 코드 모음"** 셀을 **가장 먼저 실행**해.

---

## 📁 부록 — 오늘의 코드 모음 (제일 먼저 실행!)

In [ ]:
from typing import MutableSequence, Sequence
import sys, random, time, math
sys.setrecursionlimit(100000)

# ---------- 21일차: 기본 퀵 정렬 (가운데 원소 피벗) ----------
def qsort_basic(a, left, right):
    pl, pr = left, right
    x = a[(left + right) // 2]
    while pl <= pr:
        while a[pl] < x: pl += 1
        while a[pr] > x: pr -= 1
        if pl <= pr:
            a[pl], a[pr] = a[pr], a[pl]
            pl += 1
            pr -= 1
    if left < pr:  qsort_basic(a, left, pr)
    if pl < right: qsort_basic(a, pl, right)

def quick_sort_basic(a):
    if len(a) > 0:
        qsort_basic(a, 0, len(a) - 1)

# ---------- 4일차 Stack (최대 크기 기록 기능 추가) ----------
class Stack:
    """오늘 실험용 스택 (max_size로 최대 적재량을 기록)"""
    def __init__(self, capacity=1024):
        self.stk = []
        self.capacity = capacity
        self.max_size = 0
    def push(self, value):
        self.stk.append(value)
        self.max_size = max(self.max_size, len(self.stk))
    def pop(self):
        return self.stk.pop()
    def is_empty(self):
        return len(self.stk) == 0
    def __len__(self):
        return len(self.stk)

# ---------- 실습 6-13: sort3 ----------
def sort3(a, idx1, idx2, idx3):
    """a[idx1], a[idx2], a[idx3]을 오름차순 정렬하고 중앙값의 인덱스를 반환"""
    if a[idx2] < a[idx1]: a[idx2], a[idx1] = a[idx1], a[idx2]
    if a[idx3] < a[idx2]: a[idx3], a[idx2] = a[idx2], a[idx3]
    if a[idx2] < a[idx1]: a[idx2], a[idx1] = a[idx1], a[idx2]
    return idx2

print("준비 완료 ✅")

---
# 🔁 [Remind] 워밍업 — 21일차 되감기

오늘의 비재귀 퀵 정렬은 **21일차 재귀 퀵 정렬과 완전히 같은 일**을 한다. 다른 건 "다음에 나눌 범위를 어디에 적어두느냐"뿐이야. 그 대조를 만들려면 21일차를 정확히 기억하고 있어야 해.

### R-1. 🟢 [손] 재귀 퀵 정렬, 분할 1회

배열 `a = [5, 8, 4, 2, 6, 1, 3, 9, 7]` (n=9), `left=0`, `right=8`.

피벗 `x = a[(0+8)//2] = a[4] = 6` 으로 **첫 번째 분할만** 손으로 수행해봐.

| 항목 | 답 |
|---|---|
| 분할이 끝났을 때 배열 | ① |
| 최종 `pl` | ② |
| 최종 `pr` | ③ |
| 다음에 나눌 두 범위 | ④ |

*(직접 손으로 추적한 뒤, 아래 셀로 확인)*

In [ ]:
a = [5, 8, 4, 2, 6, 1, 3, 9, 7]
left, right = 0, 8
pl, pr = left, right
x = a[(left + right) // 2]
print(f"피벗 x = {x}")
while pl <= pr:
    while a[pl] < x: pl += 1
    while a[pr] > x: pr -= 1
    if pl <= pr:
        a[pl], a[pr] = a[pr], a[pl]
        pl += 1
        pr -= 1
        print(f"  교환 후: {a}   pl={pl}, pr={pr}")
print(f"\n최종 배열 {a}, pl={pl}, pr={pr}")
print(f"다음 범위: ({left}, {pr}) 와 ({pl}, {right})")

### R-2. 🟡 [설명] 재귀는 사실 "숨겨진 스택"이다

14일차에 재귀 함수를 스택으로 바꾸는 걸 배웠지. 21일차 재귀 퀵 정렬 마지막 두 줄을 다시 봐:

```python
if left < pr:  qsort(a, left, pr)     # 왼쪽 그룹
if pl < right: qsort(a, pl, right)    # 오른쪽 그룹
```

- 이 두 줄이 실행될 때, **"나중에 처리할 범위"는 어디에 저장**되고 있는 거야? (변수 이름이 아니라 **메모리 구조** 이름으로 답해)
- 그렇다면 오늘 만드는 비재귀 버전에서 우리가 직접 만드는 `range` 스택은, 재귀 버전의 **무엇을 손으로 흉내 낸 것**일까?
- 재귀 깊이가 너무 깊어지면 파이썬은 어떤 에러를 내지? 그 에러 이름이 이 질문의 답을 이미 말해주고 있어.

*(여기에 답 작성)*

---
# 🧩 PART 1 — 원리 분해 (1~10번)

> 목표: **"왜 이렇게 생겼는지"를 한 줄도 남김없이 설명할 수 있게 되기.**
> 코드를 짜기 전에, 코드가 왜 그 모양인지부터.

### 1. 🟢 [손] 비재귀 퀵 정렬 — 스택의 일생 추적하기

교재 실습 6-12 (264p) 코드야. 주석 없이 다시 옮겨 적었어.

```python
def qsort(a, left, right):
    range = Stack(right - left + 1)
    range.push((left, right))

    while not range.is_empty():
        pl, pr = left, right = range.pop()
        x = a[(left + right) // 2]

        while pl <= pr:
            while a[pl] < x: pl += 1
            while a[pr] > x: pr -= 1
            if pl <= pr:
                a[pl], a[pr] = a[pr], a[pl]
                pl += 1
                pr -= 1

        if left < pr:  range.push((left, pr))
        if pl < right: range.push((pl, right))
```

배열 `[5, 8, 4, 2, 6, 1, 3, 9, 7]`에 대해 교재 [그림 6-22] (267p)를 보지 말고,
**처음 3번의 `pop` 직후 스택 상태**를 손으로 적어봐.

| 시점 | pop한 튜플 | 분할 후 push된 것들 | 스택 내용 (바닥→위) |
|---|---|---|---|
| 1회차 pop 직후 | ① | ② | ③ |
| 2회차 pop 직후 | ④ | ⑤ | ⑥ |
| 3회차 pop 직후 | ⑦ | ⑧ | ⑨ |

*(답을 적은 뒤 아래 셀로 대조)*

In [ ]:
def qsort_trace(a, left, right, verbose=True):
    """비재귀 퀵 정렬 + 스택 상태 출력"""
    rng = Stack(right - left + 1)
    rng.push((left, right))
    step = 0
    while not rng.is_empty():
        step += 1
        popped = rng.pop()
        pl, pr = left, right = popped
        x = a[(left + right) // 2]
        while pl <= pr:
            while a[pl] < x: pl += 1
            while a[pr] > x: pr -= 1
            if pl <= pr:
                a[pl], a[pr] = a[pr], a[pl]
                pl += 1
                pr -= 1
        pushed = []
        if left < pr:
            rng.push((left, pr));  pushed.append((left, pr))
        if pl < right:
            rng.push((pl, right)); pushed.append((pl, right))
        if verbose:
            print(f"{step}회차 | pop {popped} | 피벗 {x} | push {pushed}")
            print(f"        스택(바닥→위) {rng.stk}")
            print(f"        배열 {a}")
    return rng.max_size

a = [5, 8, 4, 2, 6, 1, 3, 9, 7]
qsort_trace(a, 0, len(a) - 1)
print("\n결과:", a)

### 2. 🟢 [설명] 이 한 줄을 해부하라

```python
pl, pr = left, right = range.pop()
```

교재 13행이야. 한 줄에 **대입이 두 번** 들어 있어.

- 이 줄은 몇 개의 변수에 값을 넣지? 각각 뭐가 들어가?
- 파이썬의 연쇄 대입 `a = b = expr`은 **오른쪽부터** 평가돼. 그럼 `range.pop()`은 **몇 번** 호출될까? (이게 이 문법을 쓰는 실질적 이유야)
- 아래처럼 두 줄로 나눠 쓰면 **버그**가 생기는데, 왜일까?

```python
pl, pr = range.pop()
left, right = range.pop()
```

*(여기에 답 작성)*

In [ ]:
# [실험] 연쇄 대입의 평가 순서를 눈으로 확인
calls = []
def fake_pop():
    calls.append(len(calls) + 1)
    print(f"  pop() 호출 #{len(calls)}")
    return (3, 7)

print("pl, pr = left, right = fake_pop() 실행:")
pl, pr = left, right = fake_pop()
print(f"  pl={pl}, pr={pr}, left={left}, right={right}")
print(f"  → pop() 총 호출 횟수: {len(calls)}")

### 3. 🟢 [예측] 스택이 비면 왜 끝일까

```python
while not range.is_empty():
```

교재 266p는 이렇게 설명해: 스택은 "나눠야 할 배열의 범위"를 담고 있다고.

- 스택이 **비어 있다**는 건 알고리즘 입장에서 무슨 뜻이야?
- `if left < pr:` / `if pl < right:` 두 조건이 **모두 거짓**이면 어떤 상황이지? 원소가 몇 개 남았을 때 그렇게 돼?
- 만약 이 두 `if` 조건을 지우고 **무조건 push** 하면 어떻게 될까? 재귀 버전에서 이 조건을 지웠을 때 나던 에러(21일차 6번)와 어떤 관계야?

*(예측을 먼저 쓰고, 아래 셀로 확인)*

In [ ]:
# ⚠️ 조건 없이 무조건 push 하면? (안전장치로 200스텝 제한)
def qsort_no_guard(a, left, right, limit=200):
    rng = Stack()
    rng.push((left, right))
    step = 0
    while not rng.is_empty():
        step += 1
        if step > limit:
            print(f"❗ {limit}스텝 초과 — 스택 크기 {len(rng)}, 맨 위 {rng.stk[-1]}")
            return "STOPPED"
        pl, pr = left, right = rng.pop()
        x = a[(left + right) // 2]
        while pl <= pr:
            while a[pl] < x: pl += 1
            while a[pr] > x: pr -= 1
            if pl <= pr:
                a[pl], a[pr] = a[pr], a[pl]
                pl += 1; pr -= 1
        rng.push((left, pr))     # 조건 제거
        rng.push((pl, right))    # 조건 제거
    return "DONE"

print(qsort_no_guard([5, 8, 4, 2, 6, 1, 3, 9, 7], 0, 8))

### 4. 🟡 [디버깅] 🔥 부등호 하나가 프로그램을 멈춰 세운다

`quick_sort1_non_recur.py`에서 나온 실제 코드야.

```python
while pl <= pr:
    while a[pl] < x: pl += 1
    while a[pr] > x: pr -= 1
    if pl >= pr:                          # ← 여기
        a[pl], a[pr] = a[pr], a[pl]
        pl += 1
        pr -= 1
```

이 코드는 **정렬을 틀리게 하는 게 아니라, 아예 끝나지 않아.**

- `pl < pr`인 정상 상황(= 교환해야 하는 상황)에서 이 `if`는 어떻게 될까?
- `if` 블록이 실행되지 않으면 `pl`과 `pr`은 어떻게 되지? 그럼 바깥 `while pl <= pr:`의 조건은?
- **"틀린 답이 나온다"와 "영원히 안 끝난다"** 중 어느 쪽이 디버깅하기 쉬울까? 왜 그렇게 생각해?

*(답을 적은 뒤 아래 셀 실행 — 안전장치가 걸려 있으니 겁내지 마)*

In [ ]:
def qsort_bug4(a, left, right):
    rng = Stack()
    rng.push((left, right))
    while not rng.is_empty():
        pl, pr = left, right = rng.pop()
        x = a[(left + right) // 2]
        guard = 0
        while pl <= pr:
            guard += 1
            if guard > 100:
                return f"❗ 무한 루프 감지 (pl={pl}, pr={pr}, x={x}) — 배열 {a}"
            while a[pl] < x: pl += 1
            while a[pr] > x: pr -= 1
            if pl >= pr:              # 🐛 버그
                a[pl], a[pr] = a[pr], a[pl]
                pl += 1
                pr -= 1
        if left < pr:  rng.push((left, pr))
        if pl < right: rng.push((pl, right))
    return f"DONE {a}"

print(qsort_bug4([5, 8, 4, 2, 6, 1, 3, 9, 7], 0, 8))

### 5. 🟢 [디버깅] 실행조차 못 하는 코드

같은 파일의 `if __name__ == '__main__':` 블록이야.

```python
def qsort(a: MutableSequence, left: int, right: int) -> None:
    ...

if __name__ == '__main__':
    num = int(input('원소 수를 입력하세요'))
    x = [None] * num
    for i in range(num):
        x[i] = int(input(f'x[{i}]:  '))

    qsort(x)          # ← 여기
```

- 이 줄에서 나는 에러의 **정확한 이름**과 메시지를 예측해봐.
- 21일차 재귀 버전에는 `quick_sort(a)`라는 래퍼 함수가 있었어. **왜 래퍼가 필요할까?** (힌트: 사용자는 `left`, `right`를 알아야 할 이유가 있나?)
- `num = 0`을 입력하면 래퍼가 있어도 문제가 생겨. 어디서, 왜?

*(답을 적은 뒤 확인)*

In [ ]:
def qsort_need_3(a, left, right):
    pass

try:
    qsort_need_3([1, 2, 3])
except TypeError as e:
    print("TypeError:", e)

# num = 0 일 때
empty = []
print("\nlen(empty)-1 =", len(empty) - 1, "→ 이 값이 right로 들어가면?")
try:
    print(empty[(0 + (len(empty) - 1)) // 2])
except IndexError as e:
    print("IndexError:", e)

### 6. 🟡 [실험] 🔥 push 순서만 바꿨는데 스택이 3배 차이 난다

교재 268~269p의 두 규칙이야.

> **규칙 1**: 원소 수가 **많은** 쪽의 그룹을 먼저 푸시한다
> **규칙 2**: 원소 수가 **적은** 쪽의 그룹을 먼저 푸시한다

교재 예제 `[6, 5, 4, 2, 7, 3, 1, 8]` (피벗값 2)로 두 규칙을 비교하면:

- 규칙 1 → 스택에 동시에 쌓이는 최대 개수 **①____**
- 규칙 2 → 스택에 동시에 쌓이는 최대 개수 **②____**

그리고 교재는 이렇게 못 박아: 규칙 1을 쓰면 원소 수 n에 대해 **스택 최대 크기는 log n보다 적다**.
그래서 원소가 100만 개여도 스택 크기 20이면 충분하다고.

- n = 10000일 때 log₂(10000) ≈ 13.3 이야. 두 규칙의 실측 최대 스택 크기를 **예측**해봐.
  - 규칙 1: **③____**  /  규칙 2: **④____**

*(예측을 적은 뒤 아래 셀 실행)*

In [ ]:
import math, random

def qsort_rule(a, rule):
    """rule=1: 큰 그룹 먼저 push / rule=2: 작은 그룹 먼저 push. 최대 스택 크기 반환"""
    st = Stack()
    st.push((0, len(a) - 1))
    while not st.is_empty():
        pl, pr = left, right = st.pop()
        x = a[(left + right) // 2]
        while pl <= pr:
            while a[pl] < x: pl += 1
            while a[pr] > x: pr -= 1
            if pl <= pr:
                a[pl], a[pr] = a[pr], a[pl]
                pl += 1; pr -= 1
        items = []
        if left < pr:  items.append((left, pr))
        if pl < right: items.append((pl, right))
        if len(items) == 2:
            # 크기 = 끝인덱스 - 시작인덱스
            items.sort(key=lambda t: (t[1] - t[0]), reverse=(rule == 1))
        for t in items:
            st.push(t)
    return st.max_size

# (1) 교재 268p 예제
ex = [6, 5, 4, 2, 7, 3, 1, 8]
print("교재 예제 [6,5,4,2,7,3,1,8]")
print("  규칙1(큰 그룹 먼저):", qsort_rule(ex[:], 1))
print("  규칙2(작은 그룹 먼저):", qsort_rule(ex[:], 2))

# (2) n을 키워가며
random.seed(1)
print("\n  n   | 규칙1 | 규칙2 | log2(n)")
print("------+-------+-------+--------")
for n in (100, 1000, 10000):
    m1 = m2 = 0
    for _ in range(20):
        base = [random.randint(0, n * 10) for _ in range(n)]
        m1 = max(m1, qsort_rule(base[:], 1))
        m2 = max(m2, qsort_rule(base[:], 2))
    print(f"{n:5d} | {m1:5d} | {m2:5d} | {math.log2(n):6.1f}")

### 7. 🟡 [설명] 왜 "큰 그룹을 먼저" 넣으면 스택이 작아질까

6번 실험 결과를 보면 규칙 1이 압도적이야. 그 이유를 논리로 세워보자.

스택은 **LIFO** — 마지막에 넣은 게 먼저 나와. 그러니까:

- 큰 그룹을 먼저 push하면, **다음에 꺼내지는 건 어느 쪽**이야?
- 그 그룹은 원소가 적으니 **금방 다 처리돼서 스택에서 사라져**. 반대로 규칙 2에서는 큰 그룹이 먼저 꺼내져서 계속 **더 잘게 쪼개지며 새 범위를 push**하지. 이때 스택 바닥에 깔린 작은 그룹은?
- 교재 270p: "규칙 1, 2의 경우 스택에 넣고 꺼내는 **횟수(푸시, 팝)는 같지만**, 동시에 쌓이는 데이터의 **최대 개수는 다릅니다**." — 왜 총 횟수는 같은데 최대치만 달라지는지 한 문장으로 설명해봐.
- 마지막: 규칙 1에서 최대 크기가 **log n 이하**로 보장되는 이유는? (힌트: 스택에 쌓이는 그룹의 크기는 아래로 내려갈수록 어떻게 되지?)

*(여기에 답 작성)*

### 8. 🟢 [손] `sort3` — 세 값의 중앙값 구하기

```python
def sort3(a, idx1, idx2, idx3):
    if a[idx2] < a[idx1]: a[idx2], a[idx1] = a[idx1], a[idx2]   # ①
    if a[idx3] < a[idx2]: a[idx3], a[idx2] = a[idx2], a[idx3]   # ②
    if a[idx2] < a[idx1]: a[idx2], a[idx1] = a[idx1], a[idx2]   # ③
    return idx2
```

교재 270p 예제: `a = [8, 7, 6, 5, 4, 3, 2, 1, 0]`, `sort3(a, 0, 4, 8)` 호출.

| 단계 | 비교 | 교환? | 이후 `a[0], a[4], a[8]` |
|---|---|---|---|
| ① | `a[4] < a[0]`? | | |
| ② | `a[8] < a[4]`? | | |
| ③ | `a[4] < a[0]`? | | |

- 반환값과, 그때 `a[반환값]`의 값은?
- **왜 ①과 ③이 똑같은 비교를 두 번 하지?** ③을 지우면 어떤 입력에서 틀려? 반례를 하나 만들어봐. (힌트: `[2, 3, 1]` 같은 걸 넣어보면)

*(답을 적은 뒤 아래 셀로 확인)*

In [ ]:
a = [8, 7, 6, 5, 4, 3, 2, 1, 0]
print("before:", a, "| a[0],a[4],a[8] =", a[0], a[4], a[8])
m = sort3(a, 0, 4, 8)
print("after :", a, "| a[0],a[4],a[8] =", a[0], a[4], a[8])
print("반환 인덱스 m =", m, ", 피벗값 a[m] =", a[m])

# ③을 지운 버전
def sort3_broken(a, i1, i2, i3):
    if a[i2] < a[i1]: a[i2], a[i1] = a[i1], a[i2]
    if a[i3] < a[i2]: a[i3], a[i2] = a[i2], a[i3]
    return i2

print("\n--- ③ 제거 버전 전수 검사 (0~2 세 값의 모든 순열) ---")
from itertools import permutations
for p in permutations([1, 2, 3]):
    a1, a2 = list(p), list(p)
    m1 = sort3(a1, 0, 1, 2)
    m2 = sort3_broken(a2, 0, 1, 2)
    flag = "" if a1 == a2 else "   ❌ 다름!"
    print(f"{list(p)} → 정상 {a1}(중앙값 {a1[m1]}) / 3번제거 {a2}(중앙값 {a2[m2]}){flag}")

### 9. 🟡 [설명] 피벗을 왜 `a[right-1]` 자리에 숨겨두지?

교재 270~271p **방법 2**야.

```python
m = sort3(a, pl, (pl + pr) // 2, pr)   # 맨앞·가운데·맨끝을 정렬, 중앙값 인덱스 m
x = a[m]

a[m], a[pr - 1] = a[pr - 1], a[m]      # 피벗을 '맨끝에서 두 번째'로 옮김
pl += 1                                #  왼쪽 커서: left → left + 1
pr -= 2                                #  오른쪽 커서: right → right - 2
```

`sort3`가 끝난 직후, 우리는 **세 원소에 대해 확실히 아는 사실**이 생겼어.

- `a[left]`는 피벗에 대해 어떤 관계야? (이하? 이상?)
- `a[right]`는? 그리고 피벗을 옮겨놓은 `a[right-1]`은?
- 그럼 **스캔을 시작할 필요가 없는 원소가 몇 개**지? 그래서 `pl`은 +1, `pr`은 -2가 되는 거야. 각각 어떤 원소를 건너뛰는 건지 이름을 붙여봐.
- 마지막으로: 이렇게 하면 **분할이 한쪽으로 치우치는 걸 피할 수 있다**고 교재는 말해. `[8,7,6,5,4,3,2,1,0]`에서 맨 앞 원소(8)를 피벗으로 쓰면 왜 최악이 되는지, 21일차 9번과 연결해서 설명해봐.

*(여기에 답 작성)*

### 10. 🟡 [디버깅] 🔥 정답은 맞는데 틀린 코드

`quick_sort2.py`에서 나온 실제 코드야.

```python
a[m], a[pr - 1] = a[pr - 1], a[m]
pl += 1
pr -= 1          # ← 교재는 pr -= 2
```

이 코드는 **랜덤 테스트 2000회를 전부 통과해.** 정렬 결과가 항상 맞아.
그런데도 이건 버그야. 왜일까?

- `pr -= 1`이면 오른쪽 커서는 어느 인덱스에서 출발하지? 그 자리에 있는 값은 뭐야?
- `while a[pr] > x:` 조건에서 `a[pr]`이 **피벗 자기 자신**이면 어떻게 되지? (`>`가 참일까 거짓일까)
- 그래서 결과는 맞는데, **뭐가 손해**일까? 아래 실험이 그 손해를 숫자로 보여줘.
- 💡 이런 종류의 버그를 뭐라고 부를까? (18일차 들여쓰기 버그, 19일차 `-> int` 어노테이션과 같은 계열)

*(답을 적은 뒤 아래 셀 실행)*

In [ ]:
CNT = [0]

def ins_sub(a, left, right):
    for i in range(left + 1, right + 1):
        j = i; tmp = a[i]
        while j > left and a[j - 1] > tmp:
            a[j] = a[j - 1]; j -= 1
        a[j] = tmp

def make_qsort(pr_step):
    def qsort(a, left, right):
        if right - left < 9:
            ins_sub(a, left, right)
        else:
            pl, pr = left, right
            m = sort3(a, pl, (pl + pr) // 2, pr)
            x = a[m]
            a[m], a[pr - 1] = a[pr - 1], a[m]
            pl += 1
            pr -= pr_step
            while pl <= pr:
                while a[pl] < x: CNT[0] += 1; pl += 1
                CNT[0] += 1
                while a[pr] > x: CNT[0] += 1; pr -= 1
                CNT[0] += 1
                if pl <= pr:
                    a[pl], a[pr] = a[pr], a[pl]
                    pl += 1; pr -= 1
            if left < pr:  qsort(a, left, pr)
            if pl < right: qsort(a, pl, right)
    return qsort

random.seed(0)
base = [[random.randint(0, 10000) for _ in range(300)] for _ in range(50)]
for step in (1, 2):
    CNT[0] = 0
    q = make_qsort(step)
    ok = True
    for b in base:
        a = b[:]; q(a, 0, len(a) - 1)
        if a != sorted(b): ok = False
    print(f"pr -= {step}: 정렬 {'정상 ✅' if ok else '오류 ❌'}, 비교 횟수 총 {CNT[0]:,}")

---
# 🛠️ PART 2 — 응용과 구현 (11~18번)

> 이제 원리는 다 뜯어봤어. 여기서부터는 **직접 짠다.**
> `___` 를 채우고, 셀을 실행해서 기대 출력과 맞는지 확인해.

### 11. 🟡 [빈칸] 비재귀 퀵 정렬, 처음부터 끝까지

1~3번에서 뜯어본 구조를 이제 네 손으로 완성해.

**기대 출력**
```
[0, 1, 2, 3, 3, 4, 5, 5, 6, 7, 8, 9]
랜덤 500회 검증: 실패 0회
```

In [ ]:
def qsort_nonrec(a: MutableSequence, left: int, right: int) -> None:
    """a[left] ~ a[right]를 퀵 정렬 (비재귀)"""
    rng = Stack(right - left + 1)
    rng.push(___)                      # ① 최초 범위를 넣는다

    while ___:                         # ② 언제까지 반복?
        pl, pr = left, right = ___     # ③ 범위를 꺼낸다
        x = a[(left + right) // 2]

        while pl <= pr:
            while a[pl] < x: pl += 1
            while a[pr] > x: pr -= 1
            if ___:                    # ④ 교환 조건 (4번 문제 조심!)
                a[pl], a[pr] = a[pr], a[pl]
                pl += 1
                pr -= 1

        if ___: rng.push((left, pr))   # ⑤ 왼쪽 그룹을 넣을 조건
        if ___: rng.push((pl, right))  # ⑥ 오른쪽 그룹을 넣을 조건


def quick_sort_nonrec(a: MutableSequence) -> None:
    """퀵 정렬 (비재귀) — 래퍼"""
    if ___:                            # ⑦ 5번에서 찾은 엣지케이스 방어
        qsort_nonrec(a, 0, len(a) - 1)


# --- 검증 ---
x = [5, 8, 4, 2, 6, 1, 3, 9, 7, 0, 3, 5]
quick_sort_nonrec(x)
print(x)

fail = 0
for _ in range(500):
    n = random.randint(0, 40)
    t = [random.randint(0, 30) for _ in range(n)]
    ref = sorted(t)
    quick_sort_nonrec(t)
    if t != ref: fail += 1
print(f"랜덤 500회 검증: 실패 {fail}회")

### 12. 🔴 [구현] 규칙 1을 적용해 스택을 다이어트시켜라

6~7번에서 "큰 그룹을 먼저 push"가 스택을 log n 이하로 눌러준다는 걸 확인했지.
이제 11번 코드를 **규칙 1 버전**으로 고쳐봐.

**힌트**: 두 그룹 `(left, pr)`과 `(pl, right)`의 크기를 비교해서, **큰 쪽을 먼저** `push` 하면 돼.
`if`문 하나 추가하는 정도로 끝나.

**기대 출력** (숫자는 실행마다 조금 달라도 규칙1 ≤ log₂(n) 이면 정답)
```
n=2000 | 규칙1 적용: 최대 스택 ?  | 그냥 왼쪽부터: 최대 스택 ?
정렬 결과 일치: True
```

In [ ]:
def qsort_rule1(a: MutableSequence, left: int, right: int) -> int:
    """규칙 1(큰 그룹 먼저 push) 적용 비재귀 퀵 정렬. 최대 스택 크기를 반환"""
    rng = Stack()
    rng.push((left, right))
    while not rng.is_empty():
        pl, pr = left, right = rng.pop()
        x = a[(left + right) // 2]
        while pl <= pr:
            while a[pl] < x: pl += 1
            while a[pr] > x: pr -= 1
            if pl <= pr:
                a[pl], a[pr] = a[pr], a[pl]
                pl += 1; pr -= 1

        # ↓↓↓ 여기를 완성 ↓↓↓
        has_left  = left < pr
        has_right = pl < right
        if has_left and has_right:
            if ___:                    # ① 왼쪽 그룹이 더 클 조건
                rng.push(___)          # ② 먼저 push (큰 쪽)
                rng.push(___)          # ③ 나중 push (작은 쪽)
            else:
                rng.push(___)          # ④
                rng.push(___)          # ⑤
        elif has_left:
            rng.push((left, pr))
        elif has_right:
            rng.push((pl, right))
    return rng.max_size


def qsort_naive_order(a, left, right):
    """비교용: 항상 왼쪽 그룹부터 push"""
    rng = Stack()
    rng.push((left, right))
    while not rng.is_empty():
        pl, pr = left, right = rng.pop()
        x = a[(left + right) // 2]
        while pl <= pr:
            while a[pl] < x: pl += 1
            while a[pr] > x: pr -= 1
            if pl <= pr:
                a[pl], a[pr] = a[pr], a[pl]
                pl += 1; pr -= 1
        if left < pr:  rng.push((left, pr))
        if pl < right: rng.push((pl, right))
    return rng.max_size


random.seed(7)
data = [random.randint(0, 100000) for _ in range(2000)]
a1, a2 = data[:], data[:]
m1 = qsort_rule1(a1, 0, len(a1) - 1)
m2 = qsort_naive_order(a2, 0, len(a2) - 1)
print(f"n=2000 | 규칙1 적용: 최대 스택 {m1}  | 그냥 왼쪽부터: 최대 스택 {m2}")
print("정렬 결과 일치:", a1 == a2 == sorted(data))
print(f"참고: log2(2000) = {math.log2(2000):.1f}")

### 13. 🟡 [빈칸] 개선판 퀵 정렬 (실습 6-13)

교재가 적용한 두 가지 개선:

> - 원소 수가 **9개 미만**이면 단순 삽입 정렬로 전환한다
> - 피벗 선택은 **방법 2**를 채택한다

9번에서 이해한 `pl += 1`, `pr -= 2`를 정확히 쓰는 게 포인트야 (10번 함정 주의!).

**기대 출력**
```
[0, 1, 2, 3, 3, 4, 5, 5, 6, 7, 8, 9]
랜덤 500회 검증: 실패 0회
```

In [ ]:
def insertion_sort_sub(a: MutableSequence, left: int, right: int) -> None:
    """a[left] ~ a[right]를 단순 삽입 정렬 (14번에서 이 경계 조건을 다시 다룬다)"""
    for i in range(left + 1, right + 1):
        j = i
        tmp = a[i]
        while j > left and a[j - 1] > tmp:
            a[j] = a[j - 1]
            j -= 1
        a[j] = tmp


def qsort_improved(a: MutableSequence, left: int, right: int) -> None:
    """a[left] ~ a[right]를 퀵 정렬 (9개 미만이면 삽입 정렬)"""
    if ___:                                  # ① 삽입 정렬로 전환할 조건
        insertion_sort_sub(a, left, right)
    else:
        pl = left
        pr = right
        m = sort3(a, pl, ___, pr)            # ② 가운데 인덱스
        x = a[m]

        a[m], a[pr - 1] = a[pr - 1], a[m]
        pl += ___                            # ③
        pr -= ___                            # ④ (10번 함정 주의!)
        while pl <= pr:
            while a[pl] < x: pl += 1
            while a[pr] > x: pr -= 1
            if pl <= pr:
                a[pl], a[pr] = a[pr], a[pl]
                pl += 1
                pr -= 1

        if left < pr:  qsort_improved(a, left, pr)
        if pl < right: qsort_improved(a, pl, right)


def quick_sort_improved(a: MutableSequence) -> None:
    if len(a) > 0:
        qsort_improved(a, 0, len(a) - 1)


x = [5, 8, 4, 2, 6, 1, 3, 9, 7, 0, 3, 5]
quick_sort_improved(x)
print(x)

fail = 0
for _ in range(500):
    n = random.randint(0, 60)
    t = [random.randint(0, 40) for _ in range(n)]
    ref = sorted(t)
    quick_sort_improved(t)
    if t != ref: fail += 1
print(f"랜덤 500회 검증: 실패 {fail}회")

### 14. 🔴 [디버깅] 🔥 우연히 안전한 버그

`quick_sort2.py`의 삽입 정렬은 이렇게 되어 있어 (교재 17행과 동일):

```python
def insertion_sort(a, left, right):
    for i in range(left + 1, right + 1):
        j = i
        tmp = a[i]
        while j > 0 and a[j - 1] > tmp:    # ← j > 0
            a[j] = a[j - 1]
            j -= 1
        a[j] = tmp
```

이 함수는 **`a[left] ~ a[right]` 구간만** 정렬하라는 함수야. 그런데 `while j > 0`이지.

- `left = 3`인 부분 배열을 정렬하는데 `j`가 0까지 내려갈 수 있다면, 이 함수는 **어느 영역을 건드리게** 될까?
- 아래 셀에서 `[9, 9, 9, 3, 1, 2]`의 인덱스 3~5만 정렬해보면 실제로 배열이 망가져.
- **그런데** 퀵 정렬 안에서 호출될 때는 이 버그가 절대 드러나지 않아. 왜일까?
  💡 힌트: 분할이 끝난 시점에서, `a[left]`보다 **왼쪽에 있는 원소들**은 피벗에 대해 어떤 관계지? 그럼 `a[j-1] > tmp` 조건이 성립할 수 있을까?
- 이런 "**호출 맥락 덕분에 우연히 안전한 코드**"의 위험은 뭘까? (힌트: 이 함수를 다른 데서 재사용하면?)

*(답을 적은 뒤 아래 셀 실행)*

In [ ]:
def ins_j0(a, left, right):
    """while j > 0 버전 (버그)"""
    for i in range(left + 1, right + 1):
        j = i; tmp = a[i]
        while j > 0 and a[j - 1] > tmp:
            a[j] = a[j - 1]; j -= 1
        a[j] = tmp

def ins_jleft(a, left, right):
    """while j > left 버전 (안전)"""
    for i in range(left + 1, right + 1):
        j = i; tmp = a[i]
        while j > left and a[j - 1] > tmp:
            a[j] = a[j - 1]; j -= 1
        a[j] = tmp

a = [9, 9, 9, 3, 1, 2]
b = a[:]
print("원본           :", a, "  (인덱스 3~5만 정렬하라고 요청)")
ins_j0(a, 3, 5)
ins_jleft(b, 3, 5)
print("j > 0 버전     :", a, "  ← 앞쪽 9들이 밀려났다 ❌")
print("j > left 버전  :", b, "  ← 인덱스 3~5만 정렬됨 ✅")

### 15. 🟡 [실험] 개선은 정말 효과가 있었나

교재 272p: "퀵 정렬은 원소 수가 적은 경우에는 그다지 빠른 알고리즘이 아닌 것으로 알려져 있습니다."
그래서 9개 미만은 삽입 정렬로 전환한 거지.

**측정 전에 예측부터 해봐** (n=2000, 20회 반복):

| 항목 | 기본 퀵(21일차) | 개선 퀵(오늘) | 예측 |
|---|---|---|---|
| 실행 시간 | | | ①어느 쪽이 빠를까? |
| 재귀 최대 깊이 | | | ②차이가 날까? 왜? |

*(예측을 적은 뒤 실행)*

In [ ]:
DEPTH = [0]

def insertion_sort_sub(a, left, right):
    for i in range(left + 1, right + 1):
        j = i; tmp = a[i]
        while j > left and a[j - 1] > tmp:
            a[j] = a[j - 1]; j -= 1
        a[j] = tmp


def q_basic_d(a, left, right, d=1):
    DEPTH[0] = max(DEPTH[0], d)
    pl, pr = left, right
    x = a[(left + right) // 2]
    while pl <= pr:
        while a[pl] < x: pl += 1
        while a[pr] > x: pr -= 1
        if pl <= pr:
            a[pl], a[pr] = a[pr], a[pl]; pl += 1; pr -= 1
    if left < pr:  q_basic_d(a, left, pr, d + 1)
    if pl < right: q_basic_d(a, pl, right, d + 1)

def q_impr_d(a, left, right, d=1):
    DEPTH[0] = max(DEPTH[0], d)
    if right - left < 9:
        insertion_sort_sub(a, left, right); return
    pl, pr = left, right
    m = sort3(a, pl, (pl + pr) // 2, pr); x = a[m]
    a[m], a[pr - 1] = a[pr - 1], a[m]
    pl += 1; pr -= 2
    while pl <= pr:
        while a[pl] < x: pl += 1
        while a[pr] > x: pr -= 1
        if pl <= pr:
            a[pl], a[pr] = a[pr], a[pl]; pl += 1; pr -= 1
    if left < pr:  q_impr_d(a, left, pr, d + 1)
    if pl < right: q_impr_d(a, pl, right, d + 1)

random.seed(2)
N = 2000
data = [random.randint(0, 100000) for _ in range(N)]
ref = sorted(data)

for name, f in (("기본 퀵(21일차)   ", q_basic_d), ("개선 퀵(오늘)     ", q_impr_d)):
    DEPTH[0] = 0
    t0 = time.perf_counter()
    for _ in range(20):
        a = data[:]
        f(a, 0, N - 1)
    el = time.perf_counter() - t0
    print(f"{name}: {el:.3f}s | 재귀 최대 깊이 {DEPTH[0]:3d} | 정렬 {'OK' if a == ref else 'WRONG'}")

### 16. 🟢 [구현] `sorted()` — 남이 만든 무기를 정확히 알기

교재 보충수업 6-4 (274p). 노트에 **"이거 꼭 기억해라"** 라고 표시해둔 그 부분이야.

```python
a, b = sorted([a, b])          # 두 변수를 오름차순으로 재배치
a, b, c = sorted([a, b, c])
```

먼저 개념 확인:
- `sorted()`는 in-place야, 새 객체를 반환해? 반환 타입은 항상 뭐지?
- 그럼 **튜플을 정렬**하려면 왜 2단계(`tuple(sorted(x))`)가 필요해?
- 4일차에 배운 **`.sort()` vs `sorted()`**, **`.reverse()` vs `reversed()`** 구분과 같은 원리지? 한 줄로 정리해봐.

**기대 출력**
```
1) 3 7
2) (1, 2, 3)
3) [1, 3, 4, 6, 7]
4) [7, 6, 4, 3, 1]
5) ['도', '레', '미']
6) [('현진', 88), ('현수', 92), ('범관', 95)]
7) [('범관', 95), ('현수', 92), ('현진', 88)]
```

In [ ]:
# 1) 두 변수 스와핑 없이 오름차순 재배치
p, q = 7, 3
p, q = ___                                   # ①
print("1)", p, q)

# 2) 튜플 정렬 (2단계)
t = (3, 1, 2)
t = ___                                      # ② 정렬된 '튜플'로
print("2)", t)

# 3) 오름차순 / 4) 내림차순
x = [6, 3, 7, 1, 4]
print("3)", ___)                             # ③
print("4)", ___)                             # ④ reverse 사용

# 5) 문자열 리스트
notes = ['미', '도', '레']
print("5)", ___)                             # ⑤

# 6~7) key 사용 — 점수 기준 정렬
scores = [('현수', 92), ('범관', 95), ('현진', 88)]
print("6)", ___)                             # ⑥ 점수 오름차순
print("7)", ___)                             # ⑦ 점수 내림차순

### 17. 🟡 [빈칸] 병합 정렬의 심장 — 정렬된 두 배열 합치기 (실습 6-14)

06-7 시작이야. 병합 정렬 전체를 짜기 전에, **가장 중요한 부품 하나**부터.

> 각 배열에서 주목하는 원소의 값을 비교하여 **작은 쪽의 원소를 꺼내 새로운 배열에 저장**한다.

`pa`, `pb`, `pc` 세 개의 커서를 쓴다는 게 핵심이야. (18일차 셰이커 정렬의 투 포인터, 21일차 `pl`/`pr`과 같은 계보!)

**기대 출력** (교재 [그림 6-27])
```
[1, 2, 2, 3, 4, 4, 6, 8, 9, 11, 13, 16, 21]
랜덤 300회 검증: 실패 0회
```

In [ ]:
def merge_sorted_list(a: Sequence, b: Sequence, c: MutableSequence) -> None:
    """정렬을 마친 배열 a와 b를 병합하여 c에 저장"""
    pa, pb, pc = 0, 0, 0
    na, nb, nc = len(a), len(b), len(c)

    while ___ and ___:                       # ① 둘 다 아직 남아 있는 동안
        if ___:                              # ② a 쪽을 꺼낼 조건 (18번 문제와 연결!)
            c[pc] = a[pa]; pa += 1
        else:
            c[pc] = b[pb]; pb += 1
        pc += 1

    while ___:                               # ③ a에 남은 것 쏟아붓기
        c[pc] = a[pa]; pa += 1; pc += 1

    while ___:                               # ④ b에 남은 것 쏟아붓기
        c[pc] = b[pb]; pb += 1; pc += 1


a = [2, 4, 6, 8, 11, 13]
b = [1, 2, 3, 4, 9, 16, 21]
c = [None] * (len(a) + len(b))
merge_sorted_list(a, b, c)
print(c)

fail = 0
for _ in range(300):
    xa = sorted(random.randint(0, 30) for _ in range(random.randint(0, 10)))
    xb = sorted(random.randint(0, 30) for _ in range(random.randint(0, 10)))
    xc = [None] * (len(xa) + len(xb))
    merge_sorted_list(xa, xb, xc)
    if xc != sorted(list(xa) + list(xb)): fail += 1
print(f"랜덤 300회 검증: 실패 {fail}회")

### 18. 🔴 [설명] `<=` 하나가 안정성을 결정한다

17번 ②번 빈칸에 `a[pa] <= b[pb]`를 썼어? 아니면 `a[pa] < b[pb]`?
**둘 다 정렬 결과는 똑같이 맞아.** 그런데 하나는 안정적이고 하나는 아니야.

- 값이 같을 때(`a[pa] == b[pb]`) `<=`를 쓰면 **어느 쪽 원소가 먼저** `c`에 들어가지?
- 배열 `a`가 원래 앞쪽 절반이고 `b`가 뒤쪽 절반이라면, 그게 왜 **안정성**을 의미해? (18일차 안정 정렬 정의를 다시 꺼내봐)
- 아래 실험으로 확인한 뒤, 지금까지 배운 정렬의 안정성 표를 완성해:

| 정렬 | 안정? | 오늘/이전 어디서 배웠나 |
|---|---|---|
| 버블 | ✅ | 18일차 |
| 선택 | ❌ | 19일차 |
| 삽입 | ✅ | 19일차 |
| 셸 | ①____ | 20일차 |
| 퀵 | ②____ | 21일차 |
| **병합** | **③____** | 오늘 |

- 마지막 질문: 파이썬의 `sorted()`는 **Timsort**(병합 + 삽입 하이브리드)야. `sorted()`가 **안정 정렬임을 보장**하는 이유가 이제 보이지? 한 줄로.

*(답을 적은 뒤 실행)*

In [ ]:
def merge_gen(a, b, c, strict):
    pa = pb = pc = 0
    while pa < len(a) and pb < len(b):
        take_a = (a[pa][0] < b[pb][0]) if strict else (a[pa][0] <= b[pb][0])
        if take_a: c[pc] = a[pa]; pa += 1
        else:      c[pc] = b[pb]; pb += 1
        pc += 1
    while pa < len(a): c[pc] = a[pa]; pa += 1; pc += 1
    while pb < len(b): c[pc] = b[pb]; pb += 1; pc += 1

# (값, 출신표시) — 앞쪽 절반 A, 뒤쪽 절반 B
a = [(1, 'A1'), (3, 'A2'), (5, 'A3')]
b = [(1, 'B1'), (3, 'B2'), (7, 'B3')]

for strict, label in ((False, "a[pa] <= b[pb]"), (True, "a[pa] <  b[pb]")):
    c = [None] * 6
    merge_gen(a, b, c, strict)
    print(f"{label} → {[t[1] for t in c]}   (값: {[t[0] for t in c]})")

print("\nsorted()의 안정성 확인:")
print(sorted(a + b, key=lambda t: t[0]))

---
---

# ✅ 정답 & 해설

> ⚠️ **먼저 다 풀고 내려와.** 특히 4·10·14번은 답을 보면 재미가 반으로 줄어.

---

## 🔁 Remind

### R-1
피벗 `x = a[4] = 6`.

```
초기: [5, 8, 4, 2, 6, 1, 3, 9, 7]
1회: pl은 a[1]=8에서 멈춤, pr은 a[6]=3에서 멈춤 → 교환
     [5, 3, 4, 2, 6, 1, 8, 9, 7]   pl=2, pr=5
2회: pl은 a[4]=6에서 멈춤, pr은 a[5]=1에서 멈춤 → 교환
     [5, 3, 4, 2, 1, 6, 8, 9, 7]   pl=5, pr=4
3회: pl(5) > pr(4) → 루프 종료
```

- ① `[5, 3, 4, 2, 1, 6, 8, 9, 7]`  ② `pl = 5`  ③ `pr = 4`
- ④ `(0, 4)`와 `(5, 8)` — 즉 `a[0]~a[4]`와 `a[5]~a[8]`

### R-2
- **호출 스택(call stack)**. `qsort(a, left, pr)`을 호출하면 그 아래 줄 `qsort(a, pl, right)`의 인자 `pl`, `right`는 현재 스택 프레임에 살아 있다가, 위쪽 호출이 끝나면 되살아나.
- 우리가 만드는 `range` 스택은 **호출 스택이 자동으로 해주던 일을 손으로 하는 것**. 재귀는 "스택을 공짜로 쓰는 것"이지, 스택을 안 쓰는 게 아니야.
- `RecursionError: maximum recursion depth exceeded` — 이름에 **depth(깊이)** 가 들어 있다는 것 자체가, 재귀가 쌓이는 구조(=스택)임을 말해줘.

> 💡 **핵심**: 재귀 ↔ 반복+스택은 **항상 상호 변환 가능**하다. 14일차에 배운 그대로.

---

## 🧩 PART 1 해설

### 1. 스택의 일생

| 시점 | pop | push | 스택(바닥→위) |
|---|---|---|---|
| 1회차 | `(0, 8)` | `(0,4)`, `(5,8)` | `[(0,4), (5,8)]` |
| 2회차 | `(5, 8)` | `(5,6)`, `(7,8)` | `[(0,4), (5,6), (7,8)]` |
| 3회차 | `(7, 8)` | (없음 또는 1개) | `[(0,4), (5,6)]` |

**LIFO라서 나중에 넣은 오른쪽 그룹 `(5,8)`이 먼저 나온다** — 이게 교재 [그림 6-22]의 진행 순서와 정확히 같아.

> 🔑 재귀 버전은 **왼쪽부터** 깊이 파고들었는데, 비재귀 버전은 **오른쪽부터** 처리해. 결과는 같지만 순서가 뒤집힌 이유가 바로 스택의 LIFO 성질이야.

---

### 2. `pl, pr = left, right = range.pop()`

- **4개 변수**에 값이 들어간다: `pl = left = 튜플[0]`, `pr = right = 튜플[1]`.
- 연쇄 대입 `A = B = expr`은 **expr을 한 번만 평가**하고, 그 결과를 왼쪽부터 차례로 대입해. 그러니까 `pop()`은 **딱 1번** 호출돼.
- 두 줄로 나누면 `pop()`이 **2번** 호출되므로 스택에서 **범위 두 개를 꺼내버려**. 아직 처리 안 한 범위 하나가 통째로 사라져서 정렬이 안 끝나. → 스택은 파괴적(destructive) 자료구조라는 걸 잊으면 나오는 실수.

> 🔑 **한 값을 두 벌 만들어야 할 때** 쓰는 관용구. `pl/pr`은 스캔하면서 계속 변하고, `left/right`는 **원래 범위를 기억**하는 용도. 역할이 다르니까 두 벌이 필요한 거야.

---

### 3. 스택이 비면 끝

- 스택이 비었다 = **더 이상 나눌 범위가 없다** = 정렬 완료.
- 두 조건이 모두 거짓 = 나뉜 그룹의 **원소가 1개 이하**. 원소 1개짜리는 이미 정렬된 상태니까 push할 이유가 없어.
- 조건 없이 무조건 push하면 **원소 1개짜리 범위가 계속 자기 자신을 다시 push**해서 스택이 무한히 자라. 실행 결과처럼 200스텝 만에 스택 크기가 201이 돼.
  → 재귀 버전에서 `if left < pr:`를 지웠을 때 `RecursionError`가 났던 거랑 **완전히 같은 병**이야. 형태만 스택 폭발 vs 재귀 폭발로 다를 뿐.

> 🔑 **기저 조건(base case)은 재귀에만 있는 게 아니다.** 반복+스택 버전에서도 "더 이상 쪼갤 필요 없음" 판정이 반드시 필요해.

---

### 4. 🔥 `if pl >= pr:` — 무한 루프

```
pl < pr (교환해야 하는 정상 상황)
  → if pl >= pr 은 거짓
  → 교환도 안 하고, pl += 1 / pr -= 1 도 안 함
  → pl, pr 그대로
  → 다음 반복에서 while a[pl] < x 도 즉시 멈춤 (pl은 이미 x 이상)
  → 완전히 같은 상태 반복 = 무한 루프
```

**커서를 전진시키는 유일한 코드가 `if` 블록 안에 있다**는 게 핵심이야. 조건을 뒤집으면 전진 코드가 실행 안 되고, 그럼 종료 조건에 영원히 도달 못 해.

- **`>=` vs `<=`가 만든 차이**: 원래 `pl <= pr`은 "두 커서가 아직 안 만났거나 정확히 만났다"는 뜻이라 이때만 교환·전진이 의미 있어. `>=`는 정반대로 "이미 교차했다"는 상황에서만 교환하니, 정작 필요할 때 아무 일도 안 해.
- **무한 루프 vs 틀린 답**: 사실 **무한 루프가 더 낫다.** 즉시 이상하다는 걸 알 수 있으니까. 틀린 답은 "어쩌다 맞는 입력"에서 그냥 통과해버려서, 18일차 들여쓰기 버그처럼 **한참 뒤에 발견**돼.

---

### 5. `qsort(x)`

- `TypeError: qsort() missing 2 required positional arguments: 'left' and 'right'`
- 래퍼가 필요한 이유: `left`, `right`는 **재귀/반복 내부 구현 세부사항**이야. 사용자는 "이 리스트를 정렬해줘"만 알면 되고, 인덱스 범위까지 알 필요가 없어. → **인터페이스와 구현의 분리**. `quick_sort(a)`가 `qsort(a, 0, len(a)-1)`을 대신 불러줘.
- `num = 0`이면 `len(a) - 1 = -1`이 `right`로 들어가고, `a[(0 + -1)//2] = a[-1]` → 빈 리스트라 `IndexError`. 그래서 래퍼에 `if len(a) > 0:` 가드가 필요해.
  (`(0 + -1)//2`는 파이썬 바닥 나눗셈이라 `-1`이 되는 것도 확인해둬 — 2일차 몫/나머지!)

---

### 6~7. 🔥 push 순서와 스택 크기

**실측**

| n | 규칙1(큰 그룹 먼저) | 규칙2(작은 그룹 먼저) | log₂(n) |
|---|---|---|---|
| 교재 예제(8개) | **2** | **4** | 3.0 |
| 100 | 5 | 12 | 6.6 |
| 1000 | 8 | 23 | 10.0 |
| 10000 | 10 | 31 | 13.3 |

**왜 규칙 1이 이기나**

- 큰 그룹을 먼저 push → LIFO니까 **작은 그룹이 먼저 pop**된다.
- 작은 그룹은 금방 다 쪼개져서 **스택에서 사라짐**. 그러고 나서야 큰 그룹을 꺼내니, 스택에 동시에 존재하는 항목이 적어.
- 규칙 2는 반대로 **큰 그룹을 먼저 꺼내서** 계속 쪼개고 push하는 동안, 바닥에 깔린 작은 그룹들이 **처리되지 못한 채 계속 쌓여** 있어.
- **총 push/pop 횟수는 같다**: 어차피 모든 범위를 정확히 한 번씩 넣고 꺼내니까. 달라지는 건 **동시에 살아 있는 개수(=최대 높이)** 뿐.
- **log n 이하 보장**: 규칙 1에서는 push된 그룹이 위로 올라갈수록 **크기가 절반 이하**가 돼. 크기가 매번 반으로 줄어드는 항목만 쌓이니 최대 높이는 log₂n을 못 넘어. 원소 100만 개여도 20이면 충분한 이유가 이거야.

> 🔑 **자료구조의 성능은 "무엇을 넣느냐"가 아니라 "어떤 순서로 넣느냐"에서도 갈린다.** 코드 두 줄 순서만 바꿨는데 메모리가 3배 차이 나.

---

### 8. `sort3`

```
a = [8, 7, 6, 5, 4, 3, 2, 1, 0], sort3(a, 0, 4, 8)
① a[4]=4 < a[0]=8 → 교환 → a[0]=4, a[4]=8
② a[8]=0 < a[4]=8 → 교환 → a[4]=0, a[8]=8
③ a[4]=0 < a[0]=4 → 교환 → a[0]=0, a[4]=4
반환 4, 피벗 a[4] = 4  ✅ (교재 271p 그림 6-26과 동일)
```

**왜 ①과 ③이 같은 비교를 두 번 하나**: ②에서 `a[idx2]`가 **더 작은 값으로 바뀔 수 있기** 때문이야. 그러면 ①에서 이미 맞춰놨던 `a[idx1] ≤ a[idx2]` 관계가 **깨져**. 그래서 다시 한 번 확인해야 해.

**반례**: `[2, 3, 1]`
- ① `3 < 2`? 거짓 → 그대로 `[2,3,1]`
- ② `1 < 3`? 참 → 교환 → `[2,1,3]`  ← 이제 `a[0]=2 > a[1]=1`, 관계가 깨졌다!
- ③ `1 < 2`? 참 → 교환 → `[1,2,3]` ✅
- ③을 지우면 `[2,1,3]`으로 끝나서 **중앙값이 1**로 잘못 나와. (`[3,2,1]`도 같은 이유로 실패)

> 🔑 3개짜리 정렬 네트워크는 **비교 3번이 최소**야. 2번으로는 절대 안 돼.

---

### 9. 피벗을 `a[right-1]`에 숨기는 이유

`sort3(a, left, mid, right)` 직후에 확실히 아는 것:
- `a[left] ≤ 피벗` (**피벗 이하**)
- `a[right] ≥ 피벗` (**피벗 이상**)
- 피벗을 `a[right-1]`로 옮겼으니 `a[right-1] == 피벗` (**피벗과 같음 = 이상이자 이하**)

그러니 이 **3개는 스캔할 필요가 없어.**
- `pl = left + 1` : 이미 "피벗 이하"임이 확정된 `a[left]`를 건너뜀
- `pr = right - 2` : 피벗 자신(`a[right-1]`)과 "피벗 이상"이 확정된 `a[right]`를 건너뜀

**치우침 방지**: `[8,7,6,5,4,3,2,1,0]`에서 맨 앞(8)을 피벗으로 잡으면 "8 하나 : 나머지 8개"로 쪼개져서 n번 분할이 필요 → **O(n²)**. 21일차 9번에서 재귀 깊이 5 vs 125를 봤던 그거야. `sort3`는 세 값의 중앙값을 쓰니 정렬/역정렬된 입력에서도 정확히 가운데를 골라내.

---

### 10. 🔥 `pr -= 1`

**실측: `pr -= 1` 비교 96,327회 / `pr -= 2` 비교 93,169회** (정렬 결과는 둘 다 정상)

- `pr -= 1`이면 오른쪽 커서가 `right - 1`, 즉 **피벗을 옮겨둔 자리**에서 출발해.
- `while a[pr] > x:` 에서 `a[pr] == x`이므로 `>`는 **거짓** → 즉시 멈춤. 그래서 크래시도, 오답도 안 나. 그냥 **의미 없는 비교를 한 번 더** 하고 `pl`과 교환까지 하는 헛수고를 해.
- 손해: 교재가 "스캔할 원소를 3개 줄일 수 있다"고 한 최적화가 **2개로 줄어들어**. 약 3.4% 더 많은 비교.
- 💡 이건 **"조용한 성능 버그(silent performance bug)"** — 테스트를 전부 통과하기 때문에 **테스트로는 절대 못 잡는다**. 18일차 들여쓰기 버그, 19일차 `-> int` 어노테이션과 같은 계열이지만, 그것들보다 더 악질이야. 코드를 **읽고 의도를 이해해야만** 잡힌다.

> 🔑 **"테스트 통과 = 올바른 코드"가 아니다.** 정확성(correctness)과 의도(intent)는 별개.

---

## 🛠️ PART 2 해설

### 11. 비재귀 퀵 정렬

```python
rng.push((left, right))              # ①
while not rng.is_empty():            # ②
    pl, pr = left, right = rng.pop() # ③
    ...
        if pl <= pr:                 # ④  ← 4번 버그 주의
    if left < pr:  ...               # ⑤
    if pl < right: ...               # ⑥
if len(a) > 0:                       # ⑦  ← 5번 엣지케이스
```
출력: `[0, 1, 2, 3, 3, 4, 5, 5, 6, 7, 8, 9]`, 랜덤 500회 실패 0회

---

### 12. 규칙 1 적용

```python
if (pr - left) > (right - pl):       # ① 왼쪽 그룹이 더 크면
    rng.push((left, pr))             # ② 큰 쪽 먼저
    rng.push((pl, right))            # ③ 작은 쪽 나중 → 먼저 pop됨
else:
    rng.push((pl, right))            # ④ 큰 쪽 먼저
    rng.push((left, pr))             # ⑤
```

n=2000 실측: **규칙1 적용 7 / 그냥 왼쪽부터 13** (log₂2000 ≈ 11.0).
그룹 크기 비교는 `끝인덱스 - 시작인덱스`로 하면 돼 (`+1`은 양쪽 다 붙으니 비교엔 영향 없음).

> 실무 팁: C/C++ 표준 라이브러리의 `introsort`도 이 최적화를 쓴다. **재귀 깊이·스택 사용량을 상수로 못 박는 표준 기법**이야.

---

### 13. 개선판 퀵 정렬

```python
if right - left < 9:                 # ①
m = sort3(a, pl, (pl + pr) // 2, pr) # ②
pl += 1                              # ③
pr -= 2                              # ④  ← 10번 함정
```
출력: `[0, 1, 2, 3, 3, 4, 5, 5, 6, 7, 8, 9]`, 랜덤 500회 실패 0회

---

### 14. 🔥 우연히 안전한 버그

- `left = 3`인데 `j`가 0까지 갈 수 있으면, 이 함수는 **`a[0], a[1], a[2]`까지 밀어버려.** 실측: `[9,9,9,3,1,2]`의 3~5만 정렬하랬는데 `[1,2,9,9,9,3]`이 나와 — 완전히 파괴됐어.
- **퀵 정렬 안에서는 왜 안 터지나**: `qsort_improved(a, left, right)`가 호출되는 시점은 **이미 분할이 끝난 뒤**야. 퀵 정렬의 불변식상 `a[left-1]` 이하의 모든 원소는 **`a[left]~a[right]`의 어떤 값보다도 작거나 같아**. 그러니 `a[j-1] > tmp` 조건이 `j == left`에서 **절대 참이 될 수 없어** → 경계를 넘기 전에 while이 멈춰.
- **위험**: 이 함수의 안전성이 **함수 자신이 아니라 호출자의 성질에 의존**하고 있어. 누가 이 `insertion_sort`를 다른 데서 재사용하는 순간 조용히 데이터가 깨져. 함수는 **자기 계약(a[left]~a[right]만 건드린다)을 스스로 지켜야** 해.

> 🔑 이걸 **불변식(invariant)에 기댄 코드**라고 해. 동작하지만, "왜 동작하는지"가 함수 밖에 있으면 그건 시한폭탄이야. 교재 코드조차 이런 함정을 갖고 있다는 게 오늘의 교훈.

---

### 15. 성능 실측

| 항목 | 기본 퀵(21일차) | 개선 퀵(오늘) |
|---|---|---|
| 시간 (n=2000, 20회) | 약 0.038s | 약 0.031s |
| 재귀 최대 깊이 | 23 | 17 |

- **왜 빨라지나**: 원소 9개 미만 구간에서 퀵 정렬은 `sort3` + 커서 세팅 등 **고정 오버헤드**가 이득보다 커. 작은 구간은 삽입 정렬이 압승 (거의 정렬된 상태라 이동도 적음 — 19일차!).
- **왜 깊이가 줄어드나**: 9개 미만에서 재귀를 끊어버리니, 트리의 **맨 아래 3~4단계가 통째로 잘려나가**. `log₂(9) ≈ 3.2` 만큼 줄어드는 게 맞아.

---

### 16. `sorted()`

```python
p, q = sorted([p, q])                              # ①
t = tuple(sorted(t))                               # ②
sorted(x)                                          # ③
sorted(x, reverse=True)                            # ④
sorted(notes)                                      # ⑤
sorted(scores, key=lambda s: s[1])                 # ⑥
sorted(scores, key=lambda s: s[1], reverse=True)   # ⑦
```

- `sorted()`는 **in-place가 아니고, 항상 새 `list`를 반환**해. 입력이 튜플이든 문자열이든 집합이든 결과는 리스트.
- 그래서 튜플을 "정렬"하려면 ① `sorted()`로 리스트 얻고 → ② `tuple()`로 되돌리는 **2단계**가 필요해. 튜플은 불변이라 제자리 정렬 자체가 불가능하니까 (3일차 가변/불변).
- 4일차 정리와 완전히 같은 축:

| 원본 수정 (in-place, `-> None`) | 새 객체 반환 |
|---|---|
| `.sort()` / `.reverse()` / `.remove()` | `sorted()` / `reversed()` / `[::-1]` |

> 💡 `a, b = sorted([a, b])`는 **"두 변수를 조건문 없이 크기순으로 재배치"** 하는 관용구. `if a > b: a, b = b, a` 를 한 줄로 줄인 거야.

---

### 17. `merge_sorted_list`

```python
while pa < na and pb < nb:      # ①
    if a[pa] <= b[pb]:          # ②  ← 18번! (< 도 정렬은 맞지만 불안정)
...
while pa < na:                  # ③
while pb < nb:                  # ④
```
출력: `[1, 2, 2, 3, 4, 4, 6, 8, 9, 11, 13, 16, 21]` (교재 그림 6-27 일치), 랜덤 300회 실패 0회

**구조 감상**: 세 개의 `while`이 나란히 있는데, 뒤의 두 개는 **동시에 실행될 수 없어** (①이 끝났다는 건 둘 중 하나는 이미 비었다는 뜻). 총 이동 횟수는 정확히 `na + nb` → **O(n)**. 병합 정렬이 O(n log n)인 이유의 절반이 여기 있어 (나머지 절반 log n은 쪼개는 깊이).

---

### 18. 🔥 `<=` 하나가 안정성을 결정한다

**실측**

```
a[pa] <= b[pb]  →  ['A1', 'B1', 'A2', 'B2', 'A3', 'B3']   ✅ A가 항상 먼저
a[pa] <  b[pb]  →  ['B1', 'A1', 'B2', 'A2', 'A3', 'B3']   ❌ 순서 뒤집힘
```

- 값이 같을 때 `<=`는 **a 쪽(=원래 앞쪽 절반)을 먼저** 꺼내. `<`는 조건이 거짓이 되어 **b 쪽(뒤쪽 절반)을 먼저** 꺼내지.
- 병합 정렬에서 `a`는 원본의 앞 절반, `b`는 뒤 절반이야. 같은 값이면 **원래 앞에 있던 게 앞에 남아야** 안정 정렬. 그래서 `<=`가 필수.

| 정렬 | 안정? | 이유 |
|---|---|---|
| 버블 (18일) | ✅ | 인접한 것만, 큰 경우에만 교환 |
| 선택 (19일) | ❌ | 멀리 떨어진 원소와 교환 |
| 삽입 (19일) | ✅ | 같으면 멈춰서 뒤에 삽입 |
| 셸 (20일) | ① **❌** | h칸씩 건너뛰며 멀리 교환 |
| 퀵 (21일) | ② **❌** | pl/pr이 멀리 떨어진 원소를 교환 |
| **병합 (오늘)** | ③ **✅** | 같으면 앞쪽 배열을 먼저 꺼냄 (`<=`) |

- **`sorted()`가 안정 정렬인 이유**: Timsort의 뼈대가 **병합 정렬**이고, 병합 단계에서 `<=` 규칙을 지키기 때문. 그래서 `sorted(data, key=A)` 후 `sorted(data, key=B)`처럼 **다중 기준 정렬을 순차적으로** 할 수 있는 거야 (뒤에 적용한 게 1순위).

> 🔑 오늘 배운 정렬 중 **처음으로 안정적인 O(n log n) 정렬**을 만난 거야. 이게 병합 정렬이 실무에서 살아남은 이유.

---

## 📌 핵심 3줄 요약

1. **재귀 = 숨겨진 스택.** 비재귀 퀵 정렬은 호출 스택이 자동으로 하던 "나중에 처리할 범위 기억하기"를 `range` 스택으로 직접 하는 것. 그래서 **push 순서(큰 그룹 먼저)** 라는 새 선택지가 생기고, 이걸 지키면 스택 최대 크기가 **log n 이하**로 고정된다.
2. **퀵 정렬의 개선은 "약점 정확히 때리기"다.** 피벗 치우침 → `sort3`로 중앙값 확보(+스캔 3개 절약), 작은 구간 오버헤드 → 9개 미만은 삽입 정렬. 둘 다 O(n log n)을 바꾸진 않지만 **상수를 깎는다**.
3. **테스트를 통과해도 버그일 수 있다.** `pr -= 1`(조용한 성능 손실)과 `while j > 0`(호출 맥락 덕에 우연히 안전)은 둘 다 정답을 내지만 틀린 코드. 반대로 `if pl >= pr`은 즉시 무한 루프로 **시끄럽게** 실패한다 — 그게 더 나은 실패다.

## 🗂️ 스터디 진행 가이드

- 🟢 **(R-1, 1, 2, 3, 5, 8, 16번)**: 전원 필수.
  - **1번 스택 추적**이 오늘의 기본기 — 여기서 LIFO가 왜 "오른쪽 먼저"를 만드는지 못 잡으면 뒤가 전부 흐려져
  - **16번 `sorted()`** 는 코테에서 매일 쓴다. `key`와 `reverse`는 손에 붙여둘 것
- 🟡 **(R-2, 4, 6, 7, 9, 10, 11, 13, 15, 17번)**: 팀 목표선
  - **4번 무한 루프 버그**와 **10번 `pr -= 1`** 은 현수 코드에서 실제로 나온 것 🔥
  - **6번 스택 크기 실험**이 오늘의 백미 — 코드 두 줄 순서로 메모리 3배
  - **17번 병합**은 내일(23일차) 병합 정렬 전체의 부품이니 반드시 손으로 짤 것
- 🔴 **(12, 14, 18번)**: 도전
  - **14번**이 오늘 최고 난도 🔥🔥 — "동작하는데 틀린 코드"를 불변식으로 설명할 수 있으면 한 단계 올라간 거야
  - **18번 안정성**은 면접 단골
- **금요일 코딩테스트 범위**: 🟢🟡 (R-1 ~ 17번)

## 🔗 오늘 회수된 개념들

- **4일차 Stack / 14일차 재귀→스택 변환** → 비재귀 퀵 정렬 전체 (R-2, 1, 11번)
- **13일차 기저 조건** → `if left < pr` 없으면 스택 폭발 (3번)
- **18일차 셰이커 투 포인터** → `pl`/`pr` 커서, 그리고 병합의 `pa`/`pb`/`pc` (17번)
- **18~21일차 안정성 논의** → 병합 정렬에서 처음으로 "안정 + O(n log n)" 등장 (18번)
- **19일차 삽입 정렬** → 9개 미만 전환의 근거 (13, 15번)
- **21일차 피벗 선택(깊이 5 vs 125)** → `sort3` 방법 2의 존재 이유 (9번)
- **3일차 가변/불변 · 4일차 in-place vs 새 객체** → `sorted()` vs `.sort()`, 튜플 정렬 2단계 (16번)
- **2일차 바닥 나눗셈** → `(0 + -1)//2 == -1` 엣지케이스 (5번)

---

> **다음 진도 (23일차)**: 06-7 병합 정렬 본편 (재귀 분할 + 17번 병합 조립) → 06-8 힙 정렬
> 오늘 17번을 제대로 짜뒀으면 내일은 **"쪼개는 부분"만 추가**하면 끝나.